In [ ]:
import math
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sumolib
import contextily as cx
import geopandas 
import lib.dataGenLib as dg
import json
import rtree

In [ ]:
%load_ext autoreload
%autoreload 2

# choose Trace 2, 5, 7

In [ ]:
dataset_site1_path = 'traces/'
csv_files = dg.retrieve_csv_files(dataset_site1_path)
csv_files[:], len(csv_files),csv_files[9]

In [ ]:
df = pd.read_csv("traces/trace_2.csv")
df.head(2), df.shape

In [ ]:
df['timestamp'].max(),df['timestamp'].min()

In [ ]:
df['timestamp'].astype('datetime64[ms, UTC+02:00:00]').max(),df['timestamp'].astype('datetime64[ms, UTC+02:00:00]').min()

# Find all '__Vehicle__' class dataset

In [ ]:
# load sumo net xml
net = sumolib.net.readNet("./rouen.net.xml")

In [ ]:
# get sumo Geo projection reference
p = net.getGeoProj()
p.crs 

In [ ]:
df_car = df[df['Class']=='VEHICLE'].copy()
df_human = df[df['Class']=='HUMAN'].copy()

In [ ]:
p_x_car = []
p_y_car = []
p_x_human = []
p_y_human = []

for index,row in df_car.iterrows():
    lon,lat = net.convertXY2LonLat(row['X'],row['Y'])
    x, y = p(lon,lat)
    p_x_car.append(x)
    p_y_car.append(y)

for index,row in df_human.iterrows():
    lon,lat = net.convertXY2LonLat(row['X'],row['Y'])
    x, y = p(lon,lat)
    p_x_human.append(x)
    p_y_human.append(y)

df_car['p_x']=p_x_car
df_car['p_y']=p_y_car
df_human['p_x']=p_x_human
df_human['p_y']=p_y_human

In [ ]:
fig,ax = plt.subplots(figsize=(15,15))
ax.scatter(p_x_car,p_y_car,s=0.1,c='g')
ax.scatter(p_x_human,p_y_human,s=0.1,c='r')
ax.set_xlim([x-150,x+250])
ax.set_ylim([y-150,y+200])

cx.add_basemap(
    ax,
    crs= p.crs,
    source=cx.providers.OpenStreetMap.Mapnik
)

# Filter data which is not on the selected road 

In [ ]:
# radius = 0.5
# edges = net.getNeighboringEdges(v_rx.iloc[0], v_ry.iloc[0], radius)
# # pick the closest edge
# if len(edges) > 0:
#     distancesAndEdges = sorted([(dist, edge) for edge, dist in edges], key=lambda x:x[0])
#     dist, closestEdge = distancesAndEdges[0]
    
# edges,dist,closestEdge.getID() , net.getEdge('60546306#1').getType()
row,_ = df_car.shape
edge_list = []

net = sumolib.net.readNet("./rouen_carway.net.xml")
for i in range(row):
   dist, edge = dg.on_road(df_car.iloc[i,1],df_car.iloc[i,2],net,6)
   if edge is not None:
      # print(edge)
      edge_list.append(edge.getID())
   else:
      edge_list.append(edge)
df_car['edge'] = edge_list


row,_ = df_human.shape
edge_list = []

net = sumolib.net.readNet("./rouen_humanway.net.xml")
for i in range(row):
   dist, edge = dg.on_road(df_human.iloc[i,1],df_human.iloc[i,2],net,3)
   if edge is not None:
      # print(edge)
      edge_list.append(edge.getID())
   else:
      edge_list.append(edge)
df_human['edge'] = edge_list

In [ ]:
df_car = df_car[df_car['edge'].notnull()]
df_human = df_human[df_human['edge'].notnull()]

In [ ]:
# x, y = p(lon,lat)

fig,ax = plt.subplots(figsize=(15,15))
ax.scatter(df_car['p_x'],df_car['p_y'],s=0.1,c='g')
ax.scatter(df_human['p_x'],df_human['p_y'],s=0.1,c='r')
ax.set_xlim([x-150,x+250])
ax.set_ylim([y-150,y+200])

cx.add_basemap(
    ax,
    crs= p.crs,
    source=cx.providers.OpenStreetMap.Mapnik
)

# Save new training dataset with a fixed squence length

In [ ]:
all_df = pd.concat([df_car,df_human]).reset_index(drop=True)

In [ ]:
all_df.shape, all_df.head(2)

In [ ]:
col = [
 'Speed',
 'timestamp',
 'SUMO TIME']

In [ ]:
all_df['SUMO TIME'] = np.ceil((all_df['timestamp'] - all_df['timestamp'].min())/100)

In [ ]:
all_df.sort_values(['SUMO TIME']).head(20)

In [ ]:
all_df[all_df.duplicated(keep=False)]

In [ ]:
# interpolation for incomplete squences.
c_sq_df = dg.data_padding(all_df,
                          pad_method='linear',
                          period_start=0,
                          period_stop=-1,
                          scale=1,
                          pad_columns=col)

In [ ]:
c_sq_df.head(50)

In [ ]:
c_sq_df[['X','Y','p_x','p_y']] = c_sq_df[['X','Y','p_x','p_y']].interpolate(method='linear')

In [ ]:
c_sq_df = c_sq_df.convert_dtypes().ffill().dropna(how='all').copy()
c_sq_df.head(50)

In [ ]:
# check if there are null values
c_sq_df[c_sq_df.isnull().any(axis=1)],c_sq_df.shape

In [ ]:
c_sq_df = c_sq_df.dropna(how='all')
c_sq_df.shape

In [ ]:
c_sq_df[c_sq_df.duplicated()]

In [ ]:
c_sq_df[c_sq_df['Id']==10070035]

In [ ]:
# add departure and destination 
df1 = c_sq_df.copy()
v_list = []
for k,v in c_sq_df.groupby(by=['Id']):
    v['departure'] = np.repeat(v.iloc[0,9],len(v))
    v['destination'] =  np.repeat(v.iloc[-1,9],len(v))
    v_list.append(v.copy())
new_c_sq_df = pd.concat(v_list)

In [ ]:
new_c_sq_df.shape

In [ ]:
df_car1 = new_c_sq_df[new_c_sq_df['Class']=='VEHICLE']
df_human1 = new_c_sq_df[new_c_sq_df['Class']=='HUMAN']

In [ ]:
# x, y = p(lon,lat)

fig,ax = plt.subplots(figsize=(15,15))
ax.scatter(df_car1['p_x'],df_car1['p_y'],s=0.1,c='g')
ax.scatter(df_human1['p_x'],df_human1['p_y'],s=0.1,c='r')
ax.set_xlim([x-150,x+250])
ax.set_ylim([y-150,y+200])

cx.add_basemap(
    ax,
    crs= p.crs,
    source=cx.providers.OpenStreetMap.Mapnik
)

In [ ]:
new_c_sq_df[new_c_sq_df.isnull().any(axis=1)],new_c_sq_df.shape

In [ ]:
new_c_sq_df[new_c_sq_df.duplicated()]